# Preprocessing Demo

Ниже короткая демонстрация общего preprocessing pipeline: загрузка данных, time split, anti-leak признаки и базовая подготовка train/val/test.

## Импортируем функции

Подключаем модуль `preprocessing.py`, чтобы в ноутбуке не дублировать весь код.

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path('/Users/avals282006/Desktop/project/ml_in_gamedev_project/preprocessing')))

from preprocessing import (
    DEFAULT_DATA_PATH,
    DEFAULT_TARGET,
    CRM_TARGET,
    DEFAULT_TARGETS,
    load_data,
    prepare_for_target,
    prepare_for_targets,
    time_split,
    build_feature_list,
    preprocess_splits,
    TargetTransform,
    regression_metrics,
)


## Загружаем и делим по времени

Сортируем по времени, берем последние строки и делаем разбиение `70/15/15` без shuffle.

In [2]:
df = load_data(DEFAULT_DATA_PATH)
tr, va, te = time_split(df, max_rows=60000)

print('rows:', len(df))
print('split:', len(tr), len(va), len(te))
print('train max < val min < test min:', tr['start'].max(), va['start'].min(), te['start'].min())


rows: 3438527
split: 42000 9000 9000
train max < val min < test min: 2026-04-29 01:25:18 2026-04-29 01:25:49 2026-04-29 23:13:37


## Формируем список признаков

Удаляем target, ID, служебные время-колонки и target/future-like поля.

In [3]:
feature_cols = build_feature_list(df, DEFAULT_TARGET)
print('feature cols:', len(feature_cols))
feature_cols[:15]


feature cols: 73


['duration_seconds',
 'events_count',
 'past_sessions_count',
 'past_last_session_duration_sec',
 'avg_past_session_duration_sec',
 'avg_past_5_sessions_duration_sec',
 'median_past_session_duration_sec',
 'total_past_time_sec',
 'max_past_session_duration_sec',
 'min_past_session_duration_sec',
 'last_5_total_duration_sec',
 'last_pause_sec',
 'avg_past_pause_sec',
 'is_new_active_day',
 'active_days_count']

## Запускаем preprocessing

Заполняем пропуски (numeric median, categorical unknown), получаем готовые матрицы и вектора target.

In [4]:
prep = preprocess_splits(tr, va, te, feature_cols, target_col=DEFAULT_TARGET)

print('X_train:', prep.x_train.shape)
print('X_val:', prep.x_val.shape)
print('X_test:', prep.x_test.shape)
print('num/cat:', len(prep.num_cols), len(prep.cat_cols))
print('NaN check:', int(prep.x_train.isna().sum().sum() == 0 and prep.x_val.isna().sum().sum() == 0 and prep.x_test.isna().sum().sum() == 0))


X_train: (42000, 73)
X_val: (9000, 73)
X_test: (9000, 73)
num/cat: 59 14
NaN check: 1


## Проверяем target transform

Покажем, как работают режимы `raw`, `p995`, `log1p_p995`.

In [5]:
for mode in ['raw', 'p995', 'log1p_p995']:
    t = TargetTransform(mode=mode).fit(prep.y_train)
    y_tmp = t.transform(prep.y_train[:5])
    y_back = t.inverse(y_tmp)
    print(mode, '->', y_tmp.round(3), '->', y_back.round(3))


raw -> [136.  82. 745. 314. 649.] -> [136.  82. 745. 314. 649.]
p995 -> [136.  82. 745. 314. 649.] -> [136.  82. 745. 314. 649.]
log1p_p995 -> [4.92  4.419 6.615 5.753 6.477] -> [136.  82. 745. 314. 649.]


## Быстрая проверка метрик

Считаем общий набор метрик на простом dummy-прогнозе (median train).

In [6]:
import numpy as np

y_pred = np.full_like(prep.y_test, np.median(prep.y_train), dtype=float)
m = regression_metrics(prep.y_test, y_pred)
m


{'mae': 506.3261111111111,
 'medae': 255.0,
 'p70_abs_error': 297.0,
 'p90_abs_error': 1246.300000000001,
 'r2': -0.09633236586691996,
 'product_mae': 240.298238175994,
 'engagement_risk_mae': 251.60340104848322,
 'small_mae': 193.71063115957224,
 'normal_mae': 331.73889636608345,
 'long_mae': 2102.6219221604447}

## Подготовка для двух таргетов

Используем `prepare_for_targets(...)`. Если CRM-колонки нет в parquet, pipeline сам ее построит перед split.

In [7]:
packs = prepare_for_targets(df, target_cols=DEFAULT_TARGETS, max_rows=60000)

for t_col, p in packs.items():
    print('target:', t_col)
    print('  X_train/X_val/X_test:', p.x_train.shape, p.x_val.shape, p.x_test.shape)
    print('  y_train mean:', float(p.y_train.mean()))
    print('  num/cat:', len(p.num_cols), len(p.cat_cols))

target: target_next_session_length_sec
  X_train/X_val/X_test: (42000, 73) (9000, 73) (9000, 73)
  y_train mean: 666.8925476190476
  num/cat: 59 14
target: future_sessions_mean_playtime_7d
  X_train/X_val/X_test: (42000, 73) (9000, 73) (9000, 73)
  y_train mean: 596.9950310470479
  num/cat: 59 14
